# 06_fusion_selection.ipynb
## Purpose
Hybrid fusion-and-selection (PMI gate + context-fit ranking + overlap bonus + fallbacks).

## Expected inputs
- `data/dataset_merged_with_unsup_rb_aspects_ALLCOLS.csv`

## Expected outputs
- `outputs/hybrid_output.csv (best term + top-k terms)`

## Notes
- Keys are aligned using `seg_key = comment_id__seg_id`.

In [ ]:
# (Optional) If not installed yet:
# !pip -q install sentence-transformers

import pandas as pd
import numpy as np
import re, ast, math
from collections import Counter
from pathlib import Path
from tqdm import tqdm

# 1) Config

In [ ]:
# =====================
# INPUT / OUTPUT
# =====================
IN_DIR = Path("data")
# IN_PATH = IN_DIR / "dataset_merged_unsup_rb_terms_v1.csv"
IN_PATH = IN_DIR / "dataset_merged_with_unsup_rb_aspects_ALLCOLS.csv"
OUT_DIR = Path("data/outputs_hybrid")
OUT_MAIN = OUT_DIR / "dataset_hybrid_best_terms.csv"
OUT_AUD = OUT_DIR / "audit_hybrid_best_terms.csv"

df = pd.read_csv(IN_PATH)

# =====================
# CONFIG (refactor-friendly)
# =====================
CFG = {
    # Columns
    "TEXT_COL": "seg_text",
    "CAND_COL": "aspect_terms_union",
    "UNSUP_COL": "aspect_terms_unsup",              # opsional: untuk origin/overlap bonus
    "RB_COL": "aspect_terms_rb_clean",      # opsional: untuk origin/overlap bonus

    # Text canonicalization
    "PUNCT_STRIP": ".,!?;:()[]{}\"'`“”‘’/\\|<>~@#$%^&*_+=—–-",
    "MIN_TOKEN_LEN": 2,
    "MIN_TERM_LEN": 2,

    # Stopwords
    "USE_STOPWORDS": True,
    "DROP_NUMERIC_TERMS": True,
    "DROP_SINGLE_CHAR": True,

    # PMI stage
    "PMI_TAU": 0.3,            # <-- sesuai permintaan
    "TOPK_PMI": 10,
    "FALLBACK_TOP": 3,
    "PMI_ALPHA": 0.1,
    "KEEP_NEGATIVE": False,    # jika True: tetap ambil top-k PMI walau < tau

    # SBERT stage
    "SBERT_MODEL": "distiluse-base-multilingual-cased-v2",
    "BATCH_SIZE": 256,
    "SIM_TOPK_FINAL": 3,       # sim-based top-k final untuk audit
    "SIM_FALLBACK_N": 30,      # kalau PMI kosong, ambil N kandidat awal untuk sim ranking
    "OVERLAP_BONUS": 0.03,     # bonus jika term muncul di unsup & rb sekaligus
    "TAU_SIM_MIN": None,       # opsional: contoh 0.12 untuk blank out weak
}

# =====================
# Stopwords (compact, offline-safe)
# =====================
STOP_ID = {
    "yang","dan","di","ke","dari","pada","dalam","untuk","dengan","atau","itu","ini","saya","aku","kamu","dia","mereka",
    "kita","kami","anda","kalian","nya","lah","pun","kok","sih","deh","dong","ya","yah","nah","kan","tuh","nih",
    "jadi","karena","sebab","agar","supaya","bila","jika","kalau","ketika","saat","sewaktu","hingga","sampai","sejak",
    "tanpa","bukan","tidak","nggak","gak","ga","tak","belum","sudah","udah","telah","lagi","masih","akan","bakal",
    "harus","mau","ingin","bisa","dapat","boleh","perlu","cukup","lebih","kurang","paling","sangat","amat","sekali",
    "apa","siapa","dimana","mana","kapan","bagaimana","kenapa","mengapa","berapa",
    "seperti","misal","misalnya","contoh","contohnya","dll","dst","dsb","dkk","etc","etcetera",
    "via","jg","jga","aja","doang","cuma","cm","kyk","kaya","kayak",
    "sebuah","seorang","para","beberapa","banyak","sedikit","semua","seluruh","setiap","tiap",
    "antara","oleh","terhadap","mengenai","tentang","kepada","sebagai",
    "mah","teh","weh","wkwk","wk","haha","hehe","hahaha","hehehe",
}
STOP_EN = {
    "the","a","an","and","or","but","if","then","else","when","while","of","to","in","on","at","for","from","by","with",
    "as","is","are","was","were","be","been","being","this","that","these","those","it","its","i","you","he","she","they",
    "we","me","my","your","his","her","their","our","us","them","do","does","did","done","can","could","may","might","must",
    "should","would","will","just","very","more","most","less","least","not","no","yes","so","because","about","into","over",
    "under","again","still","also","there","here","what","who","where","when","why","how","which",
    "etc","etc.","w/","w/o"
}
CUSTOM_STOP = {"yg","dgn","dr","pd","dlm","utk","krn","karna","bgt","btw","imo","idk"}

STOPSET = set()
if CFG["USE_STOPWORDS"]:
    STOPSET = {w.casefold() for w in (STOP_ID | STOP_EN | CUSTOM_STOP)}

print("Rows:", len(df))

# 2) Functions

In [ ]:
def parse_list_cell(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return [str(t).strip() for t in x if str(t).strip()]
    s = str(x).strip()
    if not s or s.lower() in {"nan","none","null","[]"}:
        return []
    if s.startswith("[") and s.endswith("]"):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, list):
                return [str(t).strip() for t in v if str(t).strip()]
        except Exception:
            pass
    if any(d in s for d in [";", "|", ","]):
        delim = ";" if ";" in s else ("|" if "|" in s else ",")
        return [p.strip() for p in s.split(delim) if p.strip()]
    return [s]

_nonword = re.compile(r"[^0-9a-zA-Z\u00C0-\u024F\u1E00-\u1EFF\s]+", flags=re.UNICODE)

def canon_text(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    s = s.replace("_", " ").replace("-", " ")
    s = s.strip().lower()
    s = _nonword.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def split_tokens_keep(s: str):
    s = canon_text(s)
    return s.split() if s else []

def clean_term(term: str):
    if term is None or (isinstance(term, float) and np.isnan(term)):
        return None
    raw = str(term).strip()
    if not raw:
        return None

    toks = split_tokens_keep(raw)
    if not toks:
        return None

    kept = []
    for t in toks:
        core = t.strip(CFG["PUNCT_STRIP"]).strip()
        if not core:
            continue
        if CFG["DROP_SINGLE_CHAR"] and len(core) < CFG["MIN_TOKEN_LEN"]:
            continue
        if STOPSET and core.casefold() in STOPSET:
            continue
        if CFG["DROP_NUMERIC_TERMS"] and core.isdigit():
            continue
        kept.append(core)

    if not kept:
        return None

    out = " ".join(kept).strip()
    if len(out) < CFG["MIN_TERM_LEN"]:
        return None
    return out

# 3) Prepare Candidates

In [ ]:
# =====================
# Prepare candidates
# =====================
if CFG["TEXT_COL"] not in df.columns:
    raise ValueError(f"Kolom TEXT_COL={CFG['TEXT_COL']} tidak ada. Kolom tersedia: {list(df.columns)}")

if CFG["CAND_COL"] not in df.columns:
    raise ValueError(f"Kolom CAND_COL={CFG['CAND_COL']} tidak ada. Kolom tersedia: {list(df.columns)}")

df["sentence_text"] = df[CFG["TEXT_COL"]].fillna("").astype(str)
df["sent_tokens"]   = df["sentence_text"].apply(split_tokens_keep)

df["cand_terms_raw"] = df[CFG["CAND_COL"]].apply(parse_list_cell)
df["cand_terms_clean"] = df["cand_terms_raw"].apply(lambda lst: [t for t in (clean_term(x) for x in lst) if t])

# origin sets (optional, for overlap bonus/origin label)
if CFG["UNSUP_COL"] in df.columns:
    df["unsup_terms_clean"] = df[CFG["UNSUP_COL"]].apply(parse_list_cell).apply(lambda lst: {t for t in (clean_term(x) for x in lst) if t})
else:
    df["unsup_terms_clean"] = [set() for _ in range(len(df))]

if CFG["RB_COL"] in df.columns:
    df["rb_terms_clean"] = df[CFG["RB_COL"]].apply(parse_list_cell).apply(lambda lst: {t for t in (clean_term(x) for x in lst) if t})
else:
    df["rb_terms_clean"] = [set() for _ in range(len(df))]

# quick audit candidates
cand_n_raw = df["cand_terms_raw"].apply(len)
cand_n_clean = df["cand_terms_clean"].apply(len)

print("cand_nonempty_raw_rows:", int((cand_n_raw>0).sum()))
print("cand_nonempty_clean_rows:", int((cand_n_clean>0).sum()))
print("cand_empty_after_clean_rows:", int(((cand_n_raw>0) & (cand_n_clean==0)).sum()))
print("cand_mean_clean:", float(cand_n_clean.mean()))
print("cand_p95_clean:", int(np.quantile(cand_n_clean, 0.95)))

# 4) PMI Stage

In [ ]:
# =====================
# PMI STAGE
# =====================
corpus_counts = Counter(t for toks in df["sent_tokens"] for t in toks)
V = len(corpus_counts)
TOTAL = sum(corpus_counts.values())

def pmi_token(tf, len_sent, corpus_tf, alpha=0.1):
    denom_s = len_sent + alpha * max(V, 1)
    denom_c = TOTAL    + alpha * max(V, 1)
    p_w_given_s = (tf + alpha) / denom_s
    p_w         = (corpus_tf + alpha) / denom_c
    return math.log2(p_w_given_s / p_w)

_term_tok_cache = {}
def term_tokens(term: str):
    if term not in _term_tok_cache:
        _term_tok_cache[term] = split_tokens_keep(term)
    return _term_tok_cache[term]

def score_term_pmi(sent_counts: Counter, len_sent: int, term: str, alpha=0.1):
    ttoks = term_tokens(term)
    if not ttoks:
        return None
    scores = [pmi_token(sent_counts.get(w, 0), len_sent, corpus_counts.get(w, 0), alpha=alpha) for w in ttoks]
    return float(np.mean(scores))

PMI_TAU      = CFG["PMI_TAU"]
TOPK_PMI     = CFG["TOPK_PMI"]
FALLBACK_TOP = CFG["FALLBACK_TOP"]
ALPHA        = CFG["PMI_ALPHA"]
KEEP_NEG     = CFG["KEEP_NEGATIVE"]

def rank_terms_by_pmi(row):
    terms = row["cand_terms_clean"]
    if not terms:
        return [], [], "no_candidates"

    toks = row["sent_tokens"]
    len_sent = len(toks) if len(toks) > 0 else 1
    sent_counts = Counter(toks)

    scored = []
    for t in terms:
        s = score_term_pmi(sent_counts, len_sent, t, alpha=ALPHA)
        if s is not None:
            scored.append((t, s))

    if not scored:
        return [], [], "no_scored"

    scored.sort(key=lambda x: x[1], reverse=True)

    kept = [(t,s) for (t,s) in scored if s >= PMI_TAU][:TOPK_PMI]

    if not kept:
        if KEEP_NEG:
            kept = scored[:min(TOPK_PMI, len(scored))]
            mode = "pmi_keep_negative_topk"
        else:
            kept = scored[:min(FALLBACK_TOP, len(scored))]
            mode = "pmi_fallback_top"
    else:
        mode = "pmi_threshold"

    return [t for t,_ in kept], [s for _,s in kept], mode

tqdm.pandas()
df["best_terms_pmi"], df["best_pmi_scores"], df["pmi_mode"] = zip(
    *df.progress_apply(rank_terms_by_pmi, axis=1)
)

print(df["pmi_mode"].value_counts(dropna=False))

# 5) SBERT SIMILARITY STAGE

In [ ]:
# =====================
# SBERT SIMILARITY STAGE
# =====================
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = SentenceTransformer(CFG["SBERT_MODEL"], device=device)

# Encode all sentences once
sent_texts = df["sentence_text"].fillna("").astype(str).tolist()
sent_emb = model.encode(
    sent_texts,
    batch_size=CFG["BATCH_SIZE"],
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# Build term vocab from cleaned candidates and encode once
term_vocab = sorted({t for terms in df["cand_terms_clean"] for t in terms})
print("unique term_vocab:", len(term_vocab))

term_emb = model.encode(
    term_vocab,
    batch_size=CFG["BATCH_SIZE"],
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

term2idx = {t:i for i,t in enumerate(term_vocab)}

# 6) Pick best by SBERT similarity

In [ ]:
# =====================
# Pick best by SBERT similarity
# =====================
TOPK_FINAL = CFG["SIM_TOPK_FINAL"]
FB_N       = CFG["SIM_FALLBACK_N"]
BONUS      = CFG["OVERLAP_BONUS"]

def pick_top_by_sbert(i: int):
    cand = df.at[i, "best_terms_pmi"]
    mode = "from_best_pmi"
    if not cand:
        cand = df.at[i, "cand_terms_clean"][:FB_N]
        mode = "rescue_from_cand_terms"
    if not cand:
        return None, None, [], [], "no_candidates"

    cand_used = [t for t in cand if t in term2idx]
    if not cand_used:
        return None, None, [], [], "no_idx"

    idx = [term2idx[t] for t in cand_used]

    # cosine similarity = dot product because embeddings normalized
    sims = term_emb[idx] @ sent_emb[i]

    # overlap bonus (if term appears in both unsup & rb lists)
    uns = df.at[i, "unsup_terms_clean"]
    rb  = df.at[i, "rb_terms_clean"]
    if isinstance(uns, set) and isinstance(rb, set) and (len(uns) or len(rb)):
        for j, t in enumerate(cand_used):
            if (t in uns) and (t in rb):
                sims[j] += BONUS

    k = min(TOPK_FINAL, sims.size)
    top_pos = np.argpartition(-sims, kth=k-1)[:k]
    top_pos = top_pos[np.argsort(-sims[top_pos])]

    picked_terms = [cand_used[j] for j in top_pos.tolist()]
    picked_sims  = [float(sims[j]) for j in top_pos.tolist()]

    best_term = picked_terms[0] if picked_terms else None
    best_sim  = picked_sims[0] if picked_sims else None
    return best_term, best_sim, picked_terms, picked_sims, mode

out = [pick_top_by_sbert(i) for i in tqdm(range(len(df)), desc="SBERT similarity ranking")]
df["hybrid_best_term"], df["hybrid_best_score"], df["hybrid_topk_terms"], df["hybrid_topk_scores"], df["sim_mode"] = zip(*out)

# optional similarity thresholding
if CFG["TAU_SIM_MIN"] is not None:
    tau = float(CFG["TAU_SIM_MIN"])
    weak = df["hybrid_best_score"].fillna(-1) < tau
    df.loc[weak, ["hybrid_best_term","hybrid_best_score"]] = [None, None]

def origin_label(row):
    t = row["hybrid_best_term"]
    if not t or pd.isna(t):
        return "none"
    u = row["unsup_terms_clean"]
    r = row["rb_terms_clean"]
    in_u = (t in u) if isinstance(u, set) else False
    in_r = (t in r) if isinstance(r, set) else False
    if in_u and in_r:
        return "overlap"
    if in_u:
        return "unsup_only"
    if in_r:
        return "rb_only"
    return "unknown"

df["hybrid_best_origin"] = df.apply(origin_label, axis=1)

df[["comment_id","seg_id","sentence_text","hybrid_best_term","hybrid_best_score","hybrid_best_origin"]].head(10)

# 7) AUDIT + SAVE

In [ ]:
# =====================
# AUDIT + SAVE
# =====================


audit_rows = []

# toy audit (if toys exist)
key_cols = [c for c in ["comment_id","seg_id"] if c in df.columns]
if len(key_cols) == 2:
    audit_rows.append(("unique_keys", df[key_cols].drop_duplicates().shape[0]))
    audit_rows.append(("dup_key_rows", int(df.duplicated(subset=key_cols, keep=False).sum())))
else:
    audit_rows.append(("unique_keys", np.nan))
    audit_rows.append(("dup_key_rows", np.nan))

audit_rows.append(("rows", len(df)))
audit_rows.append(("hybrid_best_term_nonnull", int(df["hybrid_best_term"].notna().sum())))
audit_rows.append(("hybrid_best_term_null", int(df["hybrid_best_term"].isna().sum())))
audit_rows.append(("origin_overlap", int((df["hybrid_best_origin"]=="overlap").sum())))
audit_rows.append(("origin_unsup_only", int((df["hybrid_best_origin"]=="unsup_only").sum())))
audit_rows.append(("origin_rb_only", int((df["hybrid_best_origin"]=="rb_only").sum())))

scores = df["hybrid_best_score"].dropna().astype(float)
if len(scores):
    audit_rows.extend([
        ("score_mean", float(scores.mean())),
        ("score_p50", float(np.quantile(scores, 0.50))),
        ("score_p90", float(np.quantile(scores, 0.90))),
        ("score_p95", float(np.quantile(scores, 0.95))),
        ("score_min", float(scores.min())),
        ("score_max", float(scores.max())),
    ])

audit_df = pd.DataFrame(audit_rows, columns=["metric","value"])

# Save
df.to_csv(OUT_MAIN, index=False, encoding="utf-8-sig")
audit_df.to_csv(OUT_AUD, index=False, encoding="utf-8-sig")

print("Saved:", OUT_MAIN.resolve())
print("Saved:", OUT_AUD.resolve())
display(audit_df)